In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # picks up .env from the repo root
from openai import OpenAI

API_KEY = os.environ["NVIDIA_API_KEY"]
client = OpenAI(base_url = "https://integrate.api.nvidia.com/v1",api_key=API_KEY)

In [2]:
EMBED_MODEL = "nvidia/nemotron-3-embed-1b"   # or any NIM embedding model

def get_embedding(text: str, input_type: str = "passage"):
    """Return embedding vector for a single text."""
    response = client.embeddings.create(
        model=EMBED_MODEL,
        extra_body={"input_type": input_type},  # "passage" for documents, "query" for search queries
        input=[text]          # input is a list
    )
    return response.data[0].embedding


In [3]:
import numpy as np
import chromadb
import time

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 1. Semantic search with numpy
sentences = [
    "I love programming in Python.",
    "The Python snake is a constrictor.",
    "Machine learning is fascinating.",
    "I enjoy hiking in the mountains.",
]
query = "How do you feel about coding?"

sentence_embs = [get_embedding(s) for s in sentences]
query_emb = get_embedding(query, input_type="query")


sims = [cosine_similarity(query_emb, emb) for emb in sentence_embs]
top_idx = np.argmax(sims)
print(f"Query: {query}")
print(f"Top match: {sentences[top_idx]} (similarity {sims[top_idx]:.3f})")


# 2. Same with Chroma
from chromadb import Documents, EmbeddingFunction, Embeddings

class NIMEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        return [get_embedding(text) for text in input]

    def embed_query(self, input: Documents) -> Embeddings:
        # Chroma calls this for query_texts, so queries get the query-side encoding
        return [get_embedding(text, input_type="query") for text in input]

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="test1", embedding_function=NIMEmbeddingFunction())
collection.add(documents=sentences, ids=[str(i) for i in range(len(sentences))])

results = collection.query(query_texts=[query], n_results=1)
print("Chroma top match:", results["documents"][0])


Query: How do you feel about coding?
Top match: I love programming in Python. (similarity 0.386)


/tmp/ipykernel_323348/700018841.py:40: DeprecationWarning: The class NIMEmbeddingFunction does not implement __init__. This will be required in a future version.
  collection = chroma_client.create_collection(name="test1", embedding_function=NIMEmbeddingFunction())


Chroma top match: ['I love programming in Python.']
